In [ ]:
#Name: Aashish Singh , Section: Q , Roll No.: 20

In [ ]:
"""
Problem Statement: 8
Build a Python program to implement Forward Feature Selection and 
Backward Feature Elimination techniques using a given healthcare dataset 
that contains 200 records with multiple patient-related features. The 
program should perform the following: 
i. Load the dataset into dataframe. 
ii. Apply Forward Feature Selection using a Linear regression, 
kNN models and compare the results. 
iii. Apply Backward Feature Elimination using the same machine 
learning models and analyze the results. 
iv. Compare the feature sets selected by Forward Feature 
Selection and Backward Feature Elimination. 
"""

In [9]:
#Importing libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score
import statsmodels.api as sm 

In [10]:
# -------------------------------
# Load Dataset
# -------------------------------
data = pd.read_csv("healthcare.csv")

target_col = "cholesterol"

features = data.drop(columns=[target_col])
target = data[target_col]

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.2, random_state=42
)

# -------------------------------
# Forward Selection (Custom)
# -------------------------------
def forward_select(model, Xtr, Xte, ytr, yte):
    available = list(Xtr.columns)
    chosen = []
    best_r2 = -1

    while len(available) > 0:
        temp_results = []

        for col in available:
            trial_features = chosen + [col]

            model.fit(Xtr[trial_features], ytr)
            predictions = model.predict(Xte[trial_features])
            score = r2_score(yte, predictions)

            temp_results.append((col, score))

        # sort based on score
        temp_results = sorted(temp_results, key=lambda x: x[1], reverse=True)
        best_col, new_score = temp_results[0]

        if new_score > best_r2:
            chosen.append(best_col)
            available.remove(best_col)
            best_r2 = new_score
        else:
            break

    return chosen, best_r2


# Apply Forward Selection
lin_model = LinearRegression()
knn_model = KNeighborsRegressor(n_neighbors=5)

fs_lr, score_lr = forward_select(lin_model, X_train, X_test, y_train, y_test)
fs_knn, score_knn = forward_select(knn_model, X_train, X_test, y_train, y_test)

print("\n--- Forward Selection ---")
print("LR Features:", fs_lr, "| Score:", score_lr)
print("KNN Features:", fs_knn, "| Score:", score_knn)


# -------------------------------
# Backward Elimination (p-value based)
# -------------------------------
def backward_eliminate(X, y):
    X_const = sm.add_constant(X)
    cols = list(X_const.columns)

    while True:
        reg = sm.OLS(y, X_const[cols]).fit()
        pvals = reg.pvalues

        max_p = pvals.max()
        worst = pvals.idxmax()

        if max_p > 0.05:
            cols.remove(worst)
        else:
            break

    return cols


selected_be = backward_eliminate(features, target)

print("\n--- Backward Elimination ---")
print("Selected Features:", selected_be)


# -------------------------------
# Evaluation Function
# -------------------------------
def test_model(model, feat_list):
    clean_feats = [f for f in feat_list if f != "const"]

    model.fit(X_train[clean_feats], y_train)
    preds = model.predict(X_test[clean_feats])

    return r2_score(y_test, preds)


be_lr = test_model(lin_model, selected_be)
be_knn = test_model(knn_model, selected_be)

print("\n--- Backward Scores ---")
print("LR Score:", be_lr)
print("KNN Score:", be_knn)


# -------------------------------
# Final Comparison
# -------------------------------
print("\n--- Final Comparison ---")
print("Forward LR:", fs_lr)
print("Forward KNN:", fs_knn)
print("Backward:", selected_be)


--- Forward Selection ---
LR Features: ['age', 'patient_id'] | Score: 0.9107012974613807
KNN Features: ['weight_kg'] | Score: 0.39555555555555555

--- Backward Elimination ---
Selected Features: ['weight_kg', 'sugar_level']

--- Backward Scores ---
LR Score: 0.9184183439555803
KNN Score: 0.2711111111111111

--- Final Comparison ---
Forward LR: ['age', 'patient_id']
Forward KNN: ['weight_kg']
Backward: ['weight_kg', 'sugar_level']
